# 02 — EDA: brand mapping, size parsing, and granularity decision

**目标**：把建模输入面板的两处不确定性压下去 —— brand 标签可信度 + 建模粒度选择。**不估模型**，这是 `03_demand_estimation` 的事。

**四个产出**：
1. `data/processed/brand_mapping.csv` —— 每个 UPC 的 brand + manufacturer_code + brand_source + brand_confidence
2. `reports/brand_mapping_review.md` —— brand parsing 规则、manufacturer block 核对、低置信度条目
3. `data/processed/brand_size_store_week_panel.parquet` —— 聚合面板（候选 baseline）
4. `reports/eda_summary.md` —— 推荐 baseline 粒度（UPC-store-week vs brand-size-store-week）+ 判据


In [ ]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))  # import src.*

from src.features import (
    parse_size, attach_size_fields, clean_descrip, attach_descrip_clean,
    manufacturer_code, extract_brand, BRAND_RULES,
)

PROCESSED = PROJECT_ROOT / 'data' / 'processed'
REPORTS   = PROJECT_ROOT / 'reports'
FIGURES   = REPORTS / 'figures'
for p in (PROCESSED, FIGURES): p.mkdir(parents=True, exist_ok=True)

summary_lines: list[str] = []
def log(msg: str) -> None:
    print(msg); summary_lines.append(msg)

def df_to_md(df: pd.DataFrame, index: bool = False) -> str:
    """Plain markdown table writer (avoids tabulate dependency)."""
    d = df.reset_index() if index else df.copy()
    cols = list(d.columns)
    header = '| ' + ' | '.join(str(c) for c in cols) + ' |'
    sep    = '|' + '|'.join(['---'] * len(cols)) + '|'
    rows   = ['| ' + ' | '.join(str(v) for v in row) + ' |'
              for row in d.itertuples(index=False, name=None)]
    return '\n'.join([header, sep, *rows])

log('# EDA Summary — brand mapping, size parsing, granularity')
log(f'Generated: {datetime.now().isoformat(timespec="seconds")}')
log('')


## 1. Load UPC-store-week panel + attach features

调 `src.features` 的 `attach_size_fields` / `attach_descrip_clean` / `extract_brand`，保持计算逻辑只有一份。


In [ ]:
panel = pd.read_parquet(PROCESSED / 'sku_store_week_panel.parquet')
panel['UPC_int'] = panel['UPC'].astype('int64')  # preserve numeric identity for arithmetic
log(f'## 1. Load')
log(f'- rows: {len(panel):,}, UPCs: {panel["UPC"].nunique()}, stores: {panel["STORE"].nunique()}, weeks: {panel["WEEK"].nunique()}')

# size_oz, size_kind from DESCRIP? No — from SIZE column in upccer (already joined in 01).
panel = attach_size_fields(panel, size_col='SIZE')
panel = attach_descrip_clean(panel, col='DESCRIP')

# size rounding to avoid float splits in groupby keys
panel['size_oz_rounded'] = panel['size_oz'].round(2).astype('float32')

log(f'- size_kind: {panel["size_kind"].value_counts().to_dict()}')
log(f'- OZ-parseable share: {(panel["size_kind"] == "oz").mean() * 100:.1f}%')


## 2. Manufacturer block discovery

`manufacturer_code = UPC // 100000`. DFF manual (p.9) 明确 UPC 末 5 位标识 product/item，其余高位标识厂商。


In [ ]:
panel['manufacturer_code'] = manufacturer_code(panel['UPC_int'])

upc_meta = (panel
    .drop_duplicates('UPC_int')
    [['UPC_int', 'manufacturer_code', 'DESCRIP', 'descrip_clean', 'SIZE', 'size_oz', 'size_kind']]
    .rename(columns={'UPC_int': 'UPC'})
    .reset_index(drop=True))

block_sizes = (upc_meta.groupby('manufacturer_code')
               .size().sort_values(ascending=False))
log('## 2. Manufacturer blocks')
log(f'- distinct manufacturer blocks: {len(block_sizes)}')
log(f'- blocks covering ≥10 UPCs: {(block_sizes >= 10).sum()}')
log(f'- top 10 block sizes: {block_sizes.head(10).to_dict()}')


## 3. DESCRIP brand rule audit

用 `src.features.extract_brand` 给每个 UPC 打标签，统计 coverage 与 unknown。这是 A 路线；下一节与 B（manufacturer_code）交叉。


In [ ]:
upc_meta[['brand_rule', 'brand_rule_source']] = upc_meta['DESCRIP'].apply(
    lambda s: pd.Series(extract_brand(s))
)

rule_counts = upc_meta['brand_rule'].value_counts()
unknown_mask = upc_meta['brand_rule'] == 'Unknown'
log('## 3. DESCRIP rule coverage (A)')
log(f'- brand_rule counts: {rule_counts.to_dict()}')
log(f'- unknown share: {unknown_mask.mean() * 100:.1f}%')
log(f'- sample unknown DESCRIPs (≤15):')
for d in upc_meta.loc[unknown_mask, 'DESCRIP'].dropna().unique()[:15]:
    log(f'    · {d!r}')


## 4. Cross-validate brand_rule (A) against manufacturer_code blocks (B)

每个 manufacturer_code 内部应该是单一品牌。如果 block 内 brand_rule 多数一致：
- UPC 命中规则 **且** 与 block 多数一致 → `brand_source='descrip_rule'`, `brand_confidence='high'`。
- UPC 未命中规则但 block 多数明确 → `brand_source='manufacturer_block'`, `brand_confidence='medium'`（用 block 多数覆盖）。
- UPC 命中规则但与 block 多数冲突 → `brand_source='descrip_rule'`, `brand_confidence='low'`（保留 rule 但标记）。
- 两者都未知 → `brand_source='unknown'`, `brand_confidence='low'`, `brand_final='Unknown'`。

输出 **`data/processed/brand_mapping.csv`** 供 app / src 直接加载。


In [ ]:
# block-level majority brand (ignoring Unknown)
block_majority = (upc_meta.loc[~unknown_mask]
                  .groupby('manufacturer_code')['brand_rule']
                  .agg(lambda s: s.value_counts().idxmax() if len(s) else 'Unknown')
                  .rename('block_majority_brand'))
upc_meta = upc_meta.merge(block_majority, on='manufacturer_code', how='left')
upc_meta['block_majority_brand'] = upc_meta['block_majority_brand'].fillna('Unknown')

def classify(row):
    rule, blk = row['brand_rule'], row['block_majority_brand']
    if rule != 'Unknown' and rule == blk:
        return pd.Series({'brand_final': rule, 'brand_source': 'descrip_rule',       'brand_confidence': 'high'})
    if rule == 'Unknown' and blk != 'Unknown':
        return pd.Series({'brand_final': blk,  'brand_source': 'manufacturer_block', 'brand_confidence': 'medium'})
    if rule != 'Unknown' and blk == 'Unknown':
        return pd.Series({'brand_final': rule, 'brand_source': 'descrip_rule',       'brand_confidence': 'medium'})
    if rule != 'Unknown' and rule != blk:
        return pd.Series({'brand_final': rule, 'brand_source': 'descrip_rule',       'brand_confidence': 'low'})
    return pd.Series({'brand_final': 'Unknown', 'brand_source': 'unknown', 'brand_confidence': 'low'})

upc_meta[['brand_final', 'brand_source', 'brand_confidence']] = upc_meta.apply(classify, axis=1)

log('## 4. Brand cross-validation (A × B)')
log(f'- brand_confidence: {upc_meta["brand_confidence"].value_counts().to_dict()}')
log(f'- brand_source: {upc_meta["brand_source"].value_counts().to_dict()}')
log(f'- brand_final: {upc_meta["brand_final"].value_counts().to_dict()}')
conflicts = upc_meta[(upc_meta['brand_rule'] != 'Unknown') &
                     (upc_meta['block_majority_brand'] != 'Unknown') &
                     (upc_meta['brand_rule'] != upc_meta['block_majority_brand'])]
log(f'- conflicts (rule vs block): {len(conflicts)}')
for _, r in conflicts.head(10).iterrows():
    log(f'    · UPC={r["UPC"]}  DESCRIP={r["DESCRIP"]!r}  rule={r["brand_rule"]}  block={r["block_majority_brand"]}')

# Save brand_mapping.csv (reusable by src / app)
mapping_cols = ['UPC', 'manufacturer_code', 'DESCRIP', 'descrip_clean', 'SIZE',
                'size_oz', 'size_kind', 'brand_rule', 'block_majority_brand',
                'brand_final', 'brand_source', 'brand_confidence']
brand_mapping = upc_meta[mapping_cols].copy()
brand_mapping.to_csv(PROCESSED / 'brand_mapping.csv', index=False)
log(f'- saved: data/processed/brand_mapping.csv ({len(brand_mapping)} UPCs)')


## 5. SIZE parsing audit

`size_oz` 的可用率决定了 brand-size 面板能保留多少 UPC。


In [ ]:
upc_size_kind = upc_meta['size_kind'].value_counts()
row_size_kind = panel['size_kind'].value_counts()

log('## 5. SIZE parsing')
log(f'- UPC-level size_kind: {upc_size_kind.to_dict()}')
log(f'- row-level size_kind: {row_size_kind.to_dict()}')
log(f'- UPC-level oz retention: {(upc_meta["size_kind"] == "oz").mean() * 100:.1f}%')
log(f'- row-level oz retention: {(panel["size_kind"] == "oz").mean() * 100:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
upc_size_kind.plot(kind='bar', ax=axes[0], title='UPC count by size_kind')
axes[0].set_ylabel('n UPCs')

oz_sizes = upc_meta.loc[upc_meta['size_kind'].isin(['oz','oz_bundle']), 'size_oz']
oz_sizes.hist(bins=40, ax=axes[1])
axes[1].set_title('size_oz distribution (oz + oz_bundle)')
axes[1].set_xlabel('oz')
fig.tight_layout(); fig.savefig(FIGURES / 'eda_size_parsing.png', dpi=120)
plt.show()


## 6. Price / promo / brand time series

先看基础事实：brand share、promo rate 变化、价格水平。


In [ ]:
# attach brand_final back to the full panel
panel = panel.merge(upc_meta[['UPC', 'brand_final']].rename(columns={'UPC':'UPC_int'}),
                    on='UPC_int', how='left')

weekly_promo = panel.groupby('week_start_date')['promo'].mean()
weekly_price = panel.groupby('week_start_date')['unit_price'].median()
brand_share = (panel.groupby('brand_final')['MOVE'].sum()
               .sort_values(ascending=False) / panel['MOVE'].sum() * 100)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
weekly_promo.plot(ax=axes[0], title='Weekly promo rate (all UPCs)')
axes[0].set_ylabel('share promo rows')
weekly_price.plot(ax=axes[1], title='Weekly median unit_price')
axes[1].set_ylabel('USD / unit')
brand_share.plot(kind='barh', ax=axes[2], title='Brand share (% of MOVE)')
axes[2].invert_yaxis()
fig.tight_layout(); fig.savefig(FIGURES / 'eda_price_promo_brand.png', dpi=120)
plt.show()

log('## 6. Price / promo / brand')
log(f'- brand_final share (% of MOVE): {brand_share.round(1).to_dict()}')
log(f'- overall promo rate: {panel["promo"].mean() * 100:.2f}%')


## 7. Coverage diagnostics

每个候选建模单元（UPC×store vs brand×size×store）的周观测数，决定 FE 能否稳定。


In [ ]:
upc_store_weeks = panel.groupby(['UPC_int','STORE'], observed=True).size()
brand_size_store_weeks = (panel.dropna(subset=['size_oz_rounded'])
    .groupby(['brand_final','size_oz_rounded','STORE'], observed=True).size())

log('## 7. Coverage')
log(f'- UPC×store pairs: n={len(upc_store_weeks):,}, median weeks={upc_store_weeks.median():.0f}, '
    f'p10={upc_store_weeks.quantile(.1):.0f}, p90={upc_store_weeks.quantile(.9):.0f}')
log(f'- brand×size×store triples: n={len(brand_size_store_weeks):,}, '
    f'median weeks={brand_size_store_weeks.median():.0f}, '
    f'p10={brand_size_store_weeks.quantile(.1):.0f}, p90={brand_size_store_weeks.quantile(.9):.0f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
upc_store_weeks.clip(upper=400).hist(bins=40, ax=axes[0])
axes[0].set_title('Weeks per UPC×store (clip 400)'); axes[0].set_xlabel('weeks observed')
brand_size_store_weeks.clip(upper=400).hist(bins=40, ax=axes[1])
axes[1].set_title('Weeks per brand×size×store (clip 400)'); axes[1].set_xlabel('weeks observed')
fig.tight_layout(); fig.savefig(FIGURES / 'eda_coverage.png', dpi=120)
plt.show()


## 8. Aggregation rules → brand × size_oz × store × week

**聚合键**：`brand_final × size_oz_rounded × STORE × WEEK`。

**规则**（全部以 `MOVE` 加权）：
- `quantity = Σ MOVE`；`revenue = Σ (unit_price × MOVE)`；`cost_total = Σ (unit_cost × MOVE)`
- `unit_price = revenue / quantity`；`unit_cost = cost_total / quantity`
- `weighted_profit_margin = Σ(PROFIT × MOVE) / quantity`
- `promo_any = any(promo)`；`promo_share = Σ MOVE[promo] / quantity`
- `n_upcs_in_cell`, `n_promo_upcs`, `bundle_units = nunique(QTY)`
- `unit_price_min/max/std` —— dispersion 检查
- `dominant_sale_type_all` / `dominant_promo_type`：详见 §8b

**过滤**：只保留 `size_kind ∈ {oz, oz_bundle}` 且 `brand_confidence ∈ {high, medium}` 的行（避免 Unknown brand 和 ASST/CT size 污染聚合单元）。


In [ ]:
mask = (panel['size_kind'].isin(['oz','oz_bundle'])) & (panel['brand_final'] != 'Unknown')
# keep medium+ confidence
upc_conf = upc_meta.set_index('UPC')['brand_confidence'].to_dict()
panel['brand_confidence'] = panel['UPC_int'].map(upc_conf)
mask &= panel['brand_confidence'].isin(['high', 'medium'])

src = panel.loc[mask].copy()
log('## 8. Aggregation source')
log(f'- rows entering aggregation: {len(src):,} ({len(src)/len(panel)*100:.1f}% of panel)')

src['rev_row']    = src['unit_price'] * src['MOVE']
src['cost_row']   = src['unit_cost']  * src['MOVE']
src['profit_row'] = src['PROFIT']     * src['MOVE']
src['promo_move'] = np.where(src['promo'], src['MOVE'], 0)

keys = ['brand_final', 'size_oz_rounded', 'STORE', 'WEEK']
agg = src.groupby(keys, observed=True).agg(
    quantity       = ('MOVE',        'sum'),
    revenue        = ('rev_row',     'sum'),
    cost_total     = ('cost_row',    'sum'),
    profit_move    = ('profit_row',  'sum'),
    promo_quantity = ('promo_move',  'sum'),
    promo_any      = ('promo',       'any'),
    n_upcs_in_cell = ('UPC_int',     'nunique'),
    bundle_units   = ('QTY',         'nunique'),
    unit_price_min = ('unit_price',  'min'),
    unit_price_max = ('unit_price',  'max'),
    unit_price_std = ('unit_price',  'std'),
    week_start_date= ('week_start_date', 'first'),
).reset_index()

# n_promo_upcs: unique UPC count inside promo rows per cell
n_promo = (src.loc[src['promo']].groupby(keys, observed=True)['UPC_int']
           .nunique().rename('n_promo_upcs').reset_index())
agg = agg.merge(n_promo, on=keys, how='left')
agg['n_promo_upcs'] = agg['n_promo_upcs'].fillna(0).astype('int32')

# derived
agg['unit_price']   = (agg['revenue'] / agg['quantity']).astype('float32')
agg['unit_cost']    = (agg['cost_total'] / agg['quantity']).astype('float32')
agg['weighted_profit_margin'] = (agg['profit_move'] / agg['quantity']).astype('float32')
agg['promo_share']  = (agg['promo_quantity'] / agg['quantity']).astype('float32')
agg['price_per_oz'] = (agg['unit_price'] / agg['size_oz_rounded']).astype('float32')
agg['unit_price_cv']= (agg['unit_price_std'] / agg['unit_price']).astype('float32')

log(f'- aggregated rows: {len(agg):,}')


In [ ]:
# dominant_sale_type_all: MOVE-weighted mode across all rows
# dominant_promo_type: MOVE-weighted mode within promo rows only (else 'missing')
def mode_move(sub_df, key_cols, label_col, out_name):
    grouped = (sub_df.groupby(key_cols + [label_col], observed=True)['MOVE']
               .sum().reset_index())
    idx = grouped.groupby(key_cols, observed=True)['MOVE'].idxmax()
    return (grouped.loc[idx, key_cols + [label_col]]
            .rename(columns={label_col: out_name}))

dom_all   = mode_move(src, keys, 'sale_type_clean', 'dominant_sale_type_all')
promo_src = src.loc[src['promo']]
if len(promo_src):
    dom_promo = mode_move(promo_src, keys, 'sale_type_clean', 'dominant_promo_type')
else:
    dom_promo = pd.DataFrame(columns=keys + ['dominant_promo_type'])

agg = agg.merge(dom_all,   on=keys, how='left')
agg = agg.merge(dom_promo, on=keys, how='left')
agg['dominant_promo_type'] = agg['dominant_promo_type'].fillna('missing').astype('category')
agg['dominant_sale_type_all'] = agg['dominant_sale_type_all'].astype('category')

log(f'- dominant_sale_type_all: {agg["dominant_sale_type_all"].value_counts().to_dict()}')
log(f'- dominant_promo_type: {agg["dominant_promo_type"].value_counts().to_dict()}')


In [ ]:
# cell_quality_flag (comma-joined reasons; empty string = clean)
flags = []
for _, row in agg.iterrows():
    f = []
    if row['n_upcs_in_cell'] > 1 and row['bundle_units'] > 1:
        f.append('mixed_bundle')
    if row['quantity'] < 3:
        f.append('low_quantity')
    if pd.notna(row['unit_price_cv']) and row['unit_price_cv'] > 0.30:
        f.append('high_price_dispersion')
    flags.append(','.join(f))
agg['cell_quality_flag'] = pd.Series(flags, index=agg.index).astype('string')

out_path = PROCESSED / 'brand_size_store_week_panel.parquet'
agg.to_parquet(out_path, index=False, compression='snappy')
log(f'- saved: {out_path.relative_to(PROJECT_ROOT)} ({out_path.stat().st_size/1e6:.1f} MB)')
log(f'- flagged cells: {(agg["cell_quality_flag"] != "").sum():,} '
    f'({(agg["cell_quality_flag"] != "").mean()*100:.1f}%)')


## 9. UPC vs brand-size granularity comparison

**判据**：
1. 聚合后样本规模（更大 ⇒ FE 更稳）
2. 聚合后仍保留的 price variation（CV 不应大幅压缩）
3. 每 cell 周观测数（更长 ⇒ 时间内价格变化更可识别）


In [ ]:
log('## 9. UPC vs brand-size comparison')

# UPC-level snapshot
n_upc_rows = len(panel)
upc_price_cv_within = (panel.groupby(['UPC_int','STORE'], observed=True)['unit_price']
                      .agg(lambda s: s.std()/s.mean() if s.mean() else np.nan)
                      .dropna())
upc_weeks = upc_store_weeks

# brand-size-level
n_bs_rows = len(agg)
bs_price_cv_within = (agg.groupby(['brand_final','size_oz_rounded','STORE'], observed=True)['unit_price']
                      .agg(lambda s: s.std()/s.mean() if s.mean() else np.nan)
                      .dropna())
bs_weeks = brand_size_store_weeks

comparison = pd.DataFrame({
    'metric': ['rows', 'units (pair/triple)', 'median weeks/unit',
               'median within-unit price CV'],
    'UPC × store × week':   [f'{n_upc_rows:,}', f'{len(upc_weeks):,}',
                              f'{upc_weeks.median():.0f}',
                              f'{upc_price_cv_within.median():.3f}'],
    'brand × size × store × week': [f'{n_bs_rows:,}', f'{len(bs_weeks):,}',
                                      f'{bs_weeks.median():.0f}',
                                      f'{bs_price_cv_within.median():.3f}'],
})
log(comparison.pipe(df_to_md))


## 10. Recommendation

**判据**（按 Step 3 建模需求）：
1. 样本量足够跑 SKU/pair FE + week FE
2. 价格 variation 在聚合后没有被过度压缩
3. 每单元周观测数 ≥ 20（经验阈值，够做 time dim 上的识别）
4. promo_share 能区分强促销周 vs 弱促销周
5. brand_confidence 过滤不至于损失 >25% 样本


In [ ]:
# 5-criterion check
crit = {}
crit['c1_sample_size_ok']   = n_bs_rows >= 100_000
crit['c2_price_variation_preserved'] = bs_price_cv_within.median() >= 0.5 * upc_price_cv_within.median()
crit['c3_weeks_per_unit_ok']= bs_weeks.median() >= 20
crit['c4_promo_signal_ok']  = (agg['promo_share'].between(0.01, 0.99)).mean() >= 0.05
crit['c5_brand_loss_ok']    = len(src) / len(panel) >= 0.75

passes = sum(crit.values())
rec = 'brand × size × store × week' if passes >= 3 else 'UPC × store × week'

log('## 10. Recommendation')
for k, v in crit.items(): log(f'- {k}: {v}')
log(f'- criteria passed: {passes}/5')
log(f'- **Recommended baseline panel**: `{rec}`')
log(f'- **Secondary robustness panel**: `{"UPC × store × week" if rec.startswith("brand") else "brand × size × store × week"}`')
log('')
log('**Rationale**: 用加 FE 的聚合面板作 baseline（更稳定），用 UPC 粒度作 robustness check（可识别尾部 SKU 弹性）。')

# brand_mapping_review.md
review_lines = [
    '# Brand mapping review',
    '',
    f'Generated: {datetime.now().isoformat(timespec="seconds")}',
    '',
    '## Rule set used',
    '',
    '| Regex | Brand |',
    '|---|---|',
    *[f'| `{p}` | {b} |' for p, b in BRAND_RULES],
    '',
    '## Confidence distribution',
    '',
    upc_meta['brand_confidence'].value_counts().to_frame().pipe(df_to_md, index=True),
    '',
    '## Source distribution',
    '',
    upc_meta['brand_source'].value_counts().to_frame().pipe(df_to_md, index=True),
    '',
    '## Low-confidence UPCs (first 20)',
    '',
    upc_meta[upc_meta['brand_confidence'] == 'low']
        [['UPC','manufacturer_code','DESCRIP','brand_rule','block_majority_brand','brand_final']]
        .head(20).pipe(df_to_md),
]
(REPORTS / 'brand_mapping_review.md').write_text('\n'.join(review_lines), encoding='utf-8')
log(f'- saved: reports/brand_mapping_review.md')

# eda_summary.md
(REPORTS / 'eda_summary.md').write_text('\n'.join(summary_lines), encoding='utf-8')
log(f'- saved: reports/eda_summary.md')


---

## 下一步：`03_demand_estimation.ipynb`

- 读 `brand_size_store_week_panel.parquet`（baseline）+ `sku_store_week_panel.parquet`（robustness）
- 构造 `log_quantity`、`log_unit_price`、competitor price（同 store-week 其他 brand 加权均价）、holiday flag
- OLS with SKU/pair FE + week FE；Duan smearing retransformation
- Hausman IV（同 week 其他 store 同 brand 均价）作 price endogeneity 稳健性
- 比较 baseline vs robustness 粒度的价格弹性稳定性
